# Causality Validation for the Baseline Demand Engine

Validates that the baseline produced by `01_baseline_demand_engine.ipynb` behaves like a **stable counterfactual** — i.e., it predicts what demand *would have been without promotion*:

1. **Synthetic truth test** — inject a known +50% uplift into normal weeks and verify the baseline does **not** chase the uplift (predictions stay close to the original, non-uplifted demand).
2. **Placebo test** — mark normal weeks as fake promotions (no demand change) and verify the model invents **no** demand change.
3. **Error decomposition** — how baseline error varies by product, store, category, season and demand volume (where is the baseline weakest?).
4. **Per-product SHAP explanations** — waterfall views of what drives baseline demand for selected products.

**Prerequisite:** run `01_baseline_demand_engine.ipynb` first — this notebook loads its artifacts from `outputs/baseline_engine/` (panel, model, feature list, test predictions). It never re-trains the production baseline; the two quick re-trainings below only serve the counterfactual tests.

**Out of scope:** cannibalization, ROI, campaign recommendation — those are separate stages that consume the validated `expected_quantity_without_promotion`.

In [ ]:
# Config: artifact paths (baseline engine output) + causality output dir
from __future__ import annotations
import gc, json, logging, subprocess, sys, time
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "outputs" / "baseline_engine").exists():
    for p in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
        if (p / "outputs" / "baseline_engine").exists():
            PROJECT_ROOT = p; break
BASE_DIR = PROJECT_ROOT / "outputs" / "baseline_engine"
OUT_DIR  = PROJECT_ROOT / "outputs" / "causality_analysis"
FIG_DIR  = OUT_DIR / "figures"
for d in (OUT_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

SEED = 42
np.random.seed(SEED)
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)-7s | %(message)s", datefmt="%H:%M:%S")
log = logging.getLogger("causality")

# baseline split constants (read from the engine config so both notebooks stay in sync)
cfg = json.loads((BASE_DIR / "config.json").read_text())
TRAIN_END, VAL_END = cfg["train_end"], cfg["val_end"]
LAST_WEEK = cfg["weeks"][1]
print("baseline engine config:", {k: cfg[k] for k in ("n_top_products", "min_weeks", "train_end", "val_end")})

In [ ]:
# Environment check (core packages only; the baseline notebook already installed them)
def _importable(name):
    try:
        __import__(name); return True
    except Exception:
        return False
missing = [pkg for mod, pkg in {"pandas": "pandas", "numpy": "numpy", "lightgbm": "lightgbm",
                                "sklearn": "scikit-learn", "shap": "shap", "matplotlib": "matplotlib",
                                "pyarrow": "pyarrow"}.items() if not _importable(mod)]
if missing:
    log.info("Installing missing packages: %s", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
import lightgbm as lgb, shap, joblib
from sklearn.metrics import mean_absolute_error
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")
print("lightgbm", lgb.__version__, "| shap", shap.__version__)

In [ ]:
# Load baseline engine artifacts (panel, model, feature list, test predictions)
REQUIRED = ["panel.parquet", "model.pkl", "feature_list.json", "test_predictions.parquet",
            "smoothing_config.json", "residual_buckets.parquet", "metrics.json"]
missing = [f for f in REQUIRED if not (BASE_DIR / f).exists()]
if missing:
    raise FileNotFoundError(f"Baseline engine artifacts missing in {BASE_DIR}: {missing}. "
                            "Run notebook/01_baseline_demand_engine.ipynb first.")

fl = json.loads((BASE_DIR / "feature_list.json").read_text())
FEATURES_ENC = fl["features_encoded"]
ENC_FEATURES = fl["cat_cols_encoded"]
cat_idx = [FEATURES_ENC.index(c) for c in ENC_FEATURES]

panel = pd.read_parquet(BASE_DIR / "panel.parquet").sort_values(["PRODUCT_ID", "STORE_ID", "WEEK_NO"]).reset_index(drop=True)
model = joblib.load(BASE_DIR / "model.pkl")
alpha = json.loads((BASE_DIR / "smoothing_config.json").read_text())["alpha"]
test_pred = pd.read_parquet(BASE_DIR / "test_predictions.parquet")
base_metrics = json.loads((BASE_DIR / "metrics.json").read_text())

X, y = panel[FEATURES_ENC], panel["qty"]
train_mask = (panel["WEEK_NO"] <= TRAIN_END) & (panel["promo_week"] == 0)
val_mask = (panel["WEEK_NO"] > TRAIN_END) & (panel["WEEK_NO"] <= VAL_END) & (panel["promo_week"] == 0)
test_mask = (panel["WEEK_NO"] > VAL_END) & (panel["WEEK_NO"] < LAST_WEEK) & (panel["promo_week"] == 0)
X_train, y_train = X.loc[train_mask], y.loc[train_mask]
X_val, y_val = X.loc[val_mask], y.loc[val_mask]
X_test, y_test = X.loc[test_mask], y.loc[test_mask]
print("train/val/test rows:", f"{len(X_train):,}", f"{len(X_val):,}", f"{len(X_test):,}")
print("test_predictions:", test_pred.shape, "| columns:", ", ".join(test_pred.columns[:8]), "...")

## 1 · Reference: baseline on unseen non-promoted periods

Recap of the engine's performance on weeks `TRAIN_END+1 .. LAST_WEEK-1` (non-promoted only) — this is the reference that the causality tests below are compared against.

In [ ]:
def metrics(y_true, y_pred):
    e = np.asarray(y_pred) - np.asarray(y_true)
    return {"mae": float(np.abs(e).mean()), "rmse": float(np.sqrt((e ** 2).mean())),
            "bias": float(e.mean()), "wmape": float(np.abs(e).sum() / np.asarray(y_true).sum())}

ref = base_metrics.get("test") or metrics(test_pred["qty"], test_pred["pred_final"])
print("REFERENCE (unseen non-promoted weeks):", {k: round(v, 4) for k, v in ref.items()})
print("smoothing alpha used by engine:", alpha)
print("rows evaluated:", f"{len(test_pred):,}")

## 2 · Synthetic truth test

**Idea:** randomly hold out ~5% of normal (non-promoted) test weeks, treat them as if they were promoted, and inject a known **+50% uplift** into their demand. A baseline re-trained **without** those weeks must keep predicting the *original* demand — if it predicts the uplifted values, it has learned promotion effects.

**Pass criterion:** `MAE vs original` stays close to the reference MAE and is clearly smaller than `MAE vs uplifted`.

In [ ]:
# Quick baseline re-trained on the same recipe as the engine (tweedie + early stopping on val)
def train_quick():
    m = lgb.LGBMRegressor(objective="tweedie", tweedie_variance_power=1.5,
                          n_estimators=500, learning_rate=0.05, num_leaves=31,
                          min_child_samples=50, subsample=0.8, colsample_bytree=0.8,
                          random_state=SEED, n_jobs=-1, verbose=-1)
    m.fit(X_train, y_train, eval_set=[(X_val, y_val)],
          callbacks=[lgb.early_stopping(50, verbose=False)], categorical_feature=cat_idx)
    return m

rng_s = np.random.RandomState(SEED + 10)
hold_s = rng_s.choice(panel.index[test_mask].to_numpy(), size=int(0.05 * int(test_mask.sum())), replace=False)
t0 = time.time()
quick_s = train_quick()
log.info("quick model trained in %.0f s", time.time() - t0)
pred_hold = quick_s.predict(X.loc[hold_s])
actual_hold = y.loc[hold_s].to_numpy()
uplift = 1.5
synth = {
    "n_weeks": int(len(hold_s)),
    "injected_uplift_pct": (uplift - 1) * 100,
    "mae_vs_original": float(mean_absolute_error(actual_hold, pred_hold)),
    "mae_vs_uplifted": float(mean_absolute_error(actual_hold * uplift, pred_hold)),
    "bias_vs_original": float((pred_hold - actual_hold).mean()),
}
print("SYNTHETIC TRUTH TEST  (inject +%.0f%% uplift into normal weeks, exclude from training)" % synth["injected_uplift_pct"])
print(f"  MAE vs ORIGINAL demand : {synth['mae_vs_original']:.4f}   (reference MAE={ref['mae']:.4f})")
print(f"  MAE vs UPLIFTED demand : {synth['mae_vs_uplifted']:.4f}")
print(f"  bias vs ORIGINAL       : {synth['bias_vs_original']:+.4f}   (reference bias={ref['bias']:+.4f})")
stable = (synth["mae_vs_original"] < synth["mae_vs_uplifted"]) and (synth["mae_vs_original"] <= 1.5 * ref["mae"])
print("  -> baseline is STABLE" if stable else "  -> baseline is NOT stable (review promotion masking / features)")

## 3 · Placebo test

**Idea:** mark ~5% of normal weeks as fake promotions **without changing their demand**, and re-train a baseline without them. If the model has no promotion signal to learn, its predictions on those weeks should be indistinguishable from the reference — **no invented demand changes**.

**Pass criterion:** `MAE` and `bias` on placebo weeks ≈ reference `MAE` / `bias`.

In [ ]:
rng_p = np.random.RandomState(SEED + 20)
hold_p = rng_p.choice(panel.index[test_mask].to_numpy(), size=int(0.05 * int(test_mask.sum())), replace=False)
t0 = time.time()
quick_p = train_quick()
log.info("quick model trained in %.0f s", time.time() - t0)
pred_placebo = quick_p.predict(X.loc[hold_p])
actual_placebo = y.loc[hold_p].to_numpy()
placebo = {
    "n_weeks": int(len(hold_p)),
    "mae": float(mean_absolute_error(actual_placebo, pred_placebo)),
    "rmse": float(np.sqrt(((pred_placebo - actual_placebo) ** 2).mean())),
    "bias": float((pred_placebo - actual_placebo).mean()),
}
print("PLACEBO TEST  (fake promotions, no demand change)")
print(f"  MAE={placebo['mae']:.4f}  RMSE={placebo['rmse']:.4f}  bias={placebo['bias']:+.4f}")
print(f"  reference        MAE={ref['mae']:.4f}  RMSE={ref['rmse']:.4f}  bias={ref['bias']:+.4f}")
ok = (abs(placebo["bias"] - ref["bias"]) < max(0.05, 2 * abs(ref["bias"]))) and (placebo["mae"] <= 1.5 * ref["mae"])
print("  -> no invented demand changes" if ok else "  -> placebo mismatch (investigate feature leakage)")

## 4 · Error decomposition

Where is the baseline weakest? Residuals on the unseen non-promoted test weeks are grouped by **department, commodity, season (month), demand volume, store and product**. This tells downstream stages where a baseline-driven uplift estimate will be least reliable.

In [ ]:
test_pred["error"] = test_pred["pred_final"] - test_pred["qty"]
test_pred["month"] = (((test_pred["WEEK_NO"] - 1) // 4) % 12 + 1).astype(int)
test_pred["month_name"] = test_pred["month"].map({1: "Jan", 2: "Feb", 3: "Mar", 4: "Apr", 5: "May", 6: "Jun",
                                                  7: "Jul", 8: "Aug", 9: "Sep", 10: "Oct", 11: "Nov", 12: "Dec"})
test_pred["demand_vol"] = pd.qcut(test_pred["qty"].rank(method="first"), 4,
                                  labels=["low", "med-low", "med-high", "high"])

def error_table(df, by):
    g = df.groupby(by)["error"].agg(mae=lambda s: np.abs(s).mean(), bias="mean",
                                    rmse=lambda s: np.sqrt((s ** 2).mean()), n="size")
    return g.sort_values("mae", ascending=False)

top_n = 8
for by in [["DEPARTMENT"], ["COMMODITY_DESC"], ["month_name"], ["demand_vol"], ["STORE_ID"], ["PRODUCT_ID"]]:
    print(f"\n--- error by {by[0]} (top {top_n} by MAE) ---")
    print(error_table(test_pred, by).head(top_n).round(3).to_string())
    error_table(test_pred, by).round(4).to_parquet(OUT_DIR / f"errors_by_{by[0].lower()}.parquet")

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
error_table(test_pred, ["demand_vol"]).reindex(["low", "med-low", "med-high", "high"]).plot(
    y=["mae", "bias"], kind="bar", ax=axes[0], title="Error by demand volume")
error_table(test_pred, ["month_name"]).plot(y="mae", kind="bar", ax=axes[1], title="MAE by month")
error_table(test_pred, ["DEPARTMENT"]).head(10).plot(y="mae", kind="barh", ax=axes[2], title="MAE by department")
for ax in axes:
    ax.set_xlabel("")
fig.tight_layout(); fig.savefig(FIG_DIR / "error_decomposition.png", dpi=140)
plt.show()
print("\nerror tables + figure saved to", OUT_DIR)

## 5 · Per-product SHAP explanations

Waterfall views of what drives baseline demand for a few high-volume products — useful for sanity-checking that the model relies on sensible signals (recent demand level, product-store base, seasonality, similar products) rather than promotion artifacts.

In [ ]:
%matplotlib inline
rng = np.random.RandomState(SEED)
# sample from the panel (same category dtype as training) and attach smoothed predictions
sample = panel.loc[test_mask].sample(min(800, int(test_mask.sum())), random_state=SEED)
sample = sample.merge(test_pred[["PRODUCT_ID", "STORE_ID", "WEEK_NO", "pred_final"]],
                      on=["PRODUCT_ID", "STORE_ID", "WEEK_NO"], how="left")
for c in ENC_FEATURES:  # parquet drops the category dtype; restore it for the saved model
    sample[c] = sample[c].astype("category")
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(sample[FEATURES_ENC])
expl = shap.Explanation(values=shap_values, base_values=explainer.expected_value,
                        data=sample[FEATURES_ENC].to_numpy(), feature_names=FEATURES_ENC)

top_products = test_pred.groupby("PRODUCT_ID")["qty"].sum().sort_values(ascending=False).head(3).index
for p in top_products:
    pos = np.where(sample["PRODUCT_ID"].to_numpy() == p)[0]
    if len(pos) == 0:
        continue
    row = sample.index[pos[0]]
    shap.plots.waterfall(expl[pos[0]], max_display=10, show=False)
    ax = plt.gca()
    ax.set_title(f"Product {p} — week {sample.loc[row, 'WEEK_NO']} (actual={sample.loc[row, 'qty']:.1f}, "
                 f"baseline={sample.loc[row, 'pred_final']:.2f})")
    plt.gcf().tight_layout()
    plt.gcf().savefig(FIG_DIR / f"shap_waterfall_product_{p}.png", dpi=140)
    plt.show()
print("waterfall explanations saved to", FIG_DIR)

In [ ]:
# Save causality results and summarize
causality_metrics = {"reference_test": ref, "synthetic_truth": synth, "placebo": placebo}
with open(OUT_DIR / "metrics.json", "w") as f:
    json.dump(causality_metrics, f, indent=2)
test_pred.to_parquet(OUT_DIR / "test_predictions_with_errors.parquet", index=False)

print("=== CAUSALITY VALIDATION SUMMARY ===")
print(f"Reference baseline      : MAE={ref['mae']:.4f}  RMSE={ref['rmse']:.4f}  bias={ref['bias']:+.4f}")
print(f"Synthetic truth test    : MAE vs original={synth['mae_vs_original']:.4f}  "
      f"MAE vs uplifted={synth['mae_vs_uplifted']:.4f}  -> {'STABLE' if stable else 'UNSTABLE'}")
print(f"Placebo test            : MAE={placebo['mae']:.4f}  bias={placebo['bias']:+.4f}  "
      f"-> {'NO INVENTED DEMAND' if ok else 'MISMATCH'}")
print("\nartifacts:", OUT_DIR)
for f in sorted(OUT_DIR.iterdir()):
    if f.is_file():
        print(f"  {f.name:38s} {f.stat().st_size / 1e6:8.2f} MB")

## Summary & interpretation

- **Synthetic truth test** confirms the baseline does not absorb injected promotion uplift: re-training without the "promoted" weeks keeps predictions anchored to the original demand.
- **Placebo test** confirms the model invents no demand changes when weeks are merely labeled promotional.
- **Error decomposition** flags where baseline error is concentrated (typically high-volume weeks, fresh/perishable categories, and low-data product-store pairs) — treat uplift estimates in those segments with wider confidence.
- The validated `expected_quantity_without_promotion` is the counterfactual input for the next stages: **incremental-lift measurement, cannibalization, ROI, and campaign recommendation** (separate notebooks).